# Example 4.4 / 6: Subsets That Keep the Order

Companion notebook for **4.4 Ordered Sets and Time Periods** in the AMPL lab Documentation, section "Subsets That Keep the Order". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Subsets That Keep the Order

A set computed from an ordered set is ordered only if its declaration says so. Here the data
gives all the weeks and a horizon `T`. `WEEKS` keeps the first `T` of them, and
`ordered by ALL_WEEKS` makes it an ordered set with their order, so `first` and `last` work on
it. `LATER_WEEKS`, every week but the first, is declared `ordered`: the result of `diff` keeps
the order of its left operand. `to_list()` returns each set's members:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set ALL_WEEKS ordered;
param T > 0 integer;
set WEEKS = {t in ALL_WEEKS: ord(t) <= T} ordered by ALL_WEEKS;
set LATER_WEEKS ordered = WEEKS diff {first(WEEKS)};

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["ALL_WEEKS"] = ["27sep", "04oct", "11oct", "18oct"]
ampl.param["T"] = 3

In [5]:
# run.py
print(ampl.set["WEEKS"].to_list(), ampl.get_value("last(WEEKS)"))
print(ampl.set["LATER_WEEKS"].to_list())

['27sep', '04oct', '11oct'] 11oct
['04oct', '11oct']


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set ALL_WEEKS := 27sep 04oct 11oct 18oct;
param T := 3;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
display WEEKS, LATER_WEEKS;
display last(WEEKS), first(LATER_WEEKS);

set WEEKS := 27sep 04oct 11oct;

set LATER_WEEKS := 04oct 11oct;

last(WEEKS) = 11oct
first(LATER_WEEKS) = 04oct

